# Chitin
Throughout all of my analyses, things seem to come back to chitin - whether it's a DEG, hub gene, or enriched GO term. I'm interested to dig a bit more on this and explore differences in VST expression of genes related to chitin, along with the correlation with growth.

## 0. load libraries

In [2]:
library(tidyverse) # for dplyr, stringr, ggplot
library(KEGGREST) # for KEGG pathways
library(multcompView) # for stat labels on ggplot
library(GO.db)
library(AnnotationDbi)
library(rtracklayer)
library(ggrepel)
library(broom)

## 1. load CSVs

### vst

In [35]:
# phase 1 oysters
p1.vst <- read.csv('/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase1_v_phase1/new_refGenome/deseq_res/prefilter_deseq/p1_vst.csv')%>%
  pivot_longer(cols = -X,
             names_to = 'Sample',
             values_to = 'vst') %>%
rename_with(~ "Gene", 1)

head(p1.vst)

Gene,Sample,vst
<chr>,<chr>,<dbl>
LOC144621260,B1_Nu_O03,11.21797
LOC144621260,B2_Nu_O12,11.93477
LOC144621260,B4_Nu_O32,11.22476
LOC144621260,B5_Nu_O36,11.38600
LOC144621260,B6_Nu_O47,11.49149
LOC144621260,C1_Nu_W01,11.33041


In [34]:
# phase 2 oysters
p2.vst <- read.csv('/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase2_v_phase2/deseq_res/prefilter_deseq/vst_filtered.csv')%>%
  pivot_longer(cols = -X,
             names_to = 'Sample',
             values_to = 'vst') %>%
rename_with(~ "Gene", 1)

head(p2.vst)

Gene,Sample,vst
<chr>,<chr>,<dbl>
LOC144621260,B1_B1_O01,11.62347
LOC144621260,B1_W5_O50,11.42347
LOC144621260,B2_B5_O51,11.40884
LOC144621260,B2_C4_O40,11.27515
LOC144621260,B3_B4_O41,11.00057
LOC144621260,B3_C3_O30,10.98404


In [40]:
vst.all <- rbind(p1.vst, p2.vst)
head(vst.all)

Gene,Sample,vst
<chr>,<chr>,<dbl>
LOC144621260,B1_Nu_O03,11.21797
LOC144621260,B2_Nu_O12,11.93477
LOC144621260,B4_Nu_O32,11.22476
LOC144621260,B5_Nu_O36,11.38600
LOC144621260,B6_Nu_O47,11.49149
LOC144621260,C1_Nu_W01,11.33041


### meta data

In [11]:
meta <- read.csv('/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/metaData/sample_metaData.csv') %>%
mutate(complete_trtmt  = paste0(Phase1_treatment, '_', Phase2_treatment)) %>%
dplyr::select(Sample, Phase1_temp, Phase1_DO, Phase1_treatment, Phase2_temp, Phase2_DO, Phase2_treatment, complete_trtmt)

head(meta)

,Sample,Phase1_temp,Phase1_DO,Phase1_treatment,Phase2_temp,Phase2_DO,Phase2_treatment,complete_trtmt
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,B1_B1_O01,warm,hypoxic,both,warm,hypoxic,both,both_both
2,B1_Nu_O03,warm,hypoxic,both,NA,NA,NA,both_NA
3,B1_W5_O50,warm,hypoxic,both,warm,normoxic,warm,both_warm
4,B2_B5_O51,warm,hypoxic,both,warm,hypoxic,both,both_both
5,B2_C4_O40,warm,hypoxic,both,ambient,normoxic,control,both_control
6,B2_Nu_O12,warm,hypoxic,both,NA,NA,NA,both_NA


#### growth data

In [4]:
p2_growth <- read.csv('/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/metaData/growth_phase2.1_weights.csv')

# add leading zeros to single digit tag num
p2_growth$Tag_num <- sprintf("%02d", p2_growth$Tag_num)

# make sample names same as my convention
p1_str <- str_sub(p2_growth$Phase_1_treat, 1, 1)
p2_str <- str_sub(p2_growth$Phase_2_treat, 1, 1)
p2_growth$Sample <- paste0(p1_str, p2_growth$Phase_1_rep, '_', p2_str, p2_growth$Phase_2_rep, '_', p2_growth$Tag_color, p2_growth$Tag_num)

# select the columns of interest
p2_growth <- p2_growth %>%
dplyr::select(Sample, Actual_shell_growth_mg, Actual_tissue_growth_mg, Ratio_tissue_shell_mg)

head(p2_growth)

,Sample,Actual_shell_growth_mg,Actual_tissue_growth_mg,Ratio_tissue_shell_mg
,<chr>,<dbl>,<dbl>,<chr>
1,B1_B1_O01,210.27552,120.72448,0.574125224
2,B1_B1_O02,269.01966,72.58034,0.269795672
3,B1_B2_O13,418.96764,124.23236,0.29652018
4,B1_B2_O14,285.32868,148.97132,0.522104262
5,B1_B3_O25,75.84486,20.65514,0.272334078
6,B1_B3_O26,182.09100,44.30900,0.243334377


In [12]:
p1_growth <- read.csv('/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/metaData/growth_phase1_weights.csv')

# add leading zeros to single digit tag num
p1_growth$Tag_num <- sprintf("%02d", p1_growth$Tag_num)

# make sample names same as my convention
p1_str <- str_sub(p1_growth$Phase_1_treat, 1, 1)
p1_growth$Sample <- paste0(p1_str, p1_growth$Phase_1_rep, '_Nu_', p1_growth$Tag_color, p1_growth$Tag_num)

# select the columns of interest
p1_growth <- p1_growth %>%
dplyr::select(Sample, Actual_shell_growth_mg, Actual_tissue_growth_mg, Ratio_tissue_shell_mg)

head(p1_growth)

,Sample,Actual_shell_growth_mg,Actual_tissue_growth_mg,Ratio_tissue_shell_mg
,<chr>,<dbl>,<dbl>,<chr>
1,B1_Nu_O01,42.27678,21.92322,0.518564091
2,B1_Nu_O02,63.65268,42.34732,0.665287306
3,B1_Nu_O03,77.58660,27.91340,0.359770888
4,B1_Nu_O04,157.38996,148.21004,0.941674043
5,B1_Nu_O05,-19.47582,-40.02418,2.055070339
6,B1_Nu_O06,181.29930,117.50070,0.64810344


#### merge meta and growth data

In [15]:
growth <- rbind(p1_growth, p2_growth)
metaData <- merge(meta, growth, by = 'Sample')
head(metaData)

,Sample,Phase1_temp,Phase1_DO,Phase1_treatment,Phase2_temp,Phase2_DO,Phase2_treatment,complete_trtmt,Actual_shell_growth_mg,Actual_tissue_growth_mg,Ratio_tissue_shell_mg
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>
1,B1_B1_O01,warm,hypoxic,both,warm,hypoxic,both,both_both,210.2755,120.72448,0.574125224
2,B1_Nu_O03,warm,hypoxic,both,NA,NA,NA,both_NA,77.5866,27.91340,0.359770888
3,B1_W5_O50,warm,hypoxic,both,warm,normoxic,warm,both_warm,315.8883,85.61170,0.271018901
4,B2_B5_O51,warm,hypoxic,both,warm,hypoxic,both,both_both,114.3215,44.47852,0.389065292
5,B2_C4_O40,warm,hypoxic,both,ambient,normoxic,control,both_control,164.9903,55.10972,0.33401798
6,B2_Nu_O12,warm,hypoxic,both,NA,NA,NA,both_NA,150.4230,80.37700,0.534339828


In [41]:
vst.meta <- merge(vst.all, metaData, by = 'Sample')
head(vst.meta)

,Sample,Gene,vst,Phase1_temp,Phase1_DO,Phase1_treatment,Phase2_temp,Phase2_DO,Phase2_treatment,complete_trtmt,Actual_shell_growth_mg,Actual_tissue_growth_mg,Ratio_tissue_shell_mg
,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>
1,B1_B1_O01,LOC144619518,8.750378,warm,hypoxic,both,warm,hypoxic,both,both_both,210.2755,120.7245,0.574125224
2,B1_B1_O01,LOC111132192,12.912871,warm,hypoxic,both,warm,hypoxic,both,both_both,210.2755,120.7245,0.574125224
3,B1_B1_O01,LOC111111930,10.677456,warm,hypoxic,both,warm,hypoxic,both,both_both,210.2755,120.7245,0.574125224
4,B1_B1_O01,LOC111114368,8.853079,warm,hypoxic,both,warm,hypoxic,both,both_both,210.2755,120.7245,0.574125224
5,B1_B1_O01,LOC111104515,12.163968,warm,hypoxic,both,warm,hypoxic,both,both_both,210.2755,120.7245,0.574125224
6,B1_B1_O01,LOC111131168,10.395487,warm,hypoxic,both,warm,hypoxic,both,both_both,210.2755,120.7245,0.574125224


### gene names

In [42]:
genes <- as.data.frame(import('/work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ref_files/genome/GCF_053477285.1_ASM5347728v1_genomic.gff')) %>%
dplyr::select(gene, description) %>% 
na.omit

head(genes)

,gene,description
,<chr>,<chr>
2,LOC144621260,protein O-mannosyl-transferase TMTC2-like
28,LOC144621269,uncharacterized LOC144621269
63,LOC111120925,mitochondrial amidoxime-reducing component 1-like
79,Trnae-cuc,transfer RNA glutamic acid (anticodon CUC)
82,Trnae-cuc,transfer RNA glutamic acid (anticodon CUC)
85,Trnae-cuc,transfer RNA glutamic acid (anticodon CUC)


### gene2GO

In [43]:
# read in gene ID to GO term file
gene2go <- read.csv('/work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ref_files/annotations/newRef_geneGO.csv')

# expand df so every row contains one GO and one gene ID
term2gene <- gene2go %>%
  mutate(GO_terms = strsplit(Gene.Ontology.IDs, ";")) %>%  # Split by comma, semicolon, or backtick
  unnest(GO_terms) %>%
  filter(grepl("^GO:", GO_terms)) %>%  # Keep only valid GO terms
  dplyr::select(term = GO_terms, gene = gene)

# Extract GO term descriptions
go_terms <- unique(term2gene$term)

# Get descriptions from GO.db
term2name <- data.frame(
  term = go_terms,
  name = sapply(go_terms, function(x) {
    tryCatch({
      Term(GOTERM[[x]])
    }, error = function(e) {
      NA_character_
    })
  })
)

# Remove NAs
term2name <- term2name[!is.na(term2name$name), ]

# View
head(term2name) 

,term,name
,<chr>,<chr>
GO:0005261,GO:0005261,monoatomic cation channel activity
GO:0005886,GO:0005886,plasma membrane
GO:0030001,GO:0030001,metal ion transport
GO:0098655,GO:0098655,monoatomic cation transmembrane transport
GO:0004930,GO:0004930,G protein-coupled receptor activity
GO:0007186,GO:0007186,G protein-coupled receptor signaling pathway


### all genes

phase 2 oysters

In [53]:
# get list of files
p2.files <- list.files(
    path = '/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase2_v_phase2/deseq_res/prefilter_deseq',
    pattern = '\\.csv$',
    full.names = TRUE
    )

head(p2.files)

[1] "/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase2_v_phase2/deseq_res/prefilter_deseq/bb_cc.csv"
[2] "/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase2_v_phase2/deseq_res/prefilter_deseq/bc_bb.csv"
[3] "/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase2_v_phase2/deseq_res/prefilter_deseq/bc_cc.csv"
[4] "/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase2_v_phase2/deseq_res/prefilter_deseq/bc_hc.csv"
[5] "/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase2_v_phase2/deseq_res/prefilter_deseq/bc_wc.csv"
[6] "/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase2_v_phase2/deseq_res/prefilter_deseq/bh_ch.csv"

In [ ]:
names(p2.files) <- tools::file_path_sans_ext(basename(p2.files))
p2.list <- lapply(p2.files, read.csv)
names(p2.list)

In [ ]:
head(p2.list$bb_cc)

phase 1 oysters

In [55]:
# get list of files
p1.files <- list.files(
    path = '/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase1_v_phase1/new_refGenome/deseq_res/prefilter_deseq',
    pattern = '\\.csv$',
    full.names = TRUE
    )

head(p1.files)

[1] "/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase1_v_phase1/new_refGenome/deseq_res/prefilter_deseq/p1_vst.csv"        
[2] "/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase1_v_phase1/new_refGenome/deseq_res/prefilter_deseq/p1.both_v_cont.csv"
[3] "/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase1_v_phase1/new_refGenome/deseq_res/prefilter_deseq/p1.hyp_v_both.csv" 
[4] "/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase1_v_phase1/new_refGenome/deseq_res/prefilter_deseq/p1.hyp_v_cont.csv" 
[5] "/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase1_v_phase1/new_refGenome/deseq_res/prefilter_deseq/p1.hyp_v_warm.csv" 
[6] "/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase1_v_phase1/new_refGenome/deseq_res/prefilter_deseq/p1.warm_v_both.csv"

In [56]:
names(p1.files) <- tools::file_path_sans_ext(basename(p1.files))
p1.list <- lapply(p1.files, read.csv)
names(p1.list)

[1] "p1_vst"         "p1.both_v_cont" "p1.hyp_v_both"  "p1.hyp_v_cont" 
[5] "p1.hyp_v_warm"  "p1.warm_v_both" "p1.warm_v_cont"

In [58]:
head(p1.list$p1.both_v_cont)

,X,baseMean,log2FoldChange,lfcSE,pvalue,padj
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,LOC144621260,2772.7662,0.0006761802,0.02228363,7.787130e-01,0.996744247
2,LOC144621269,6797.3814,0.0043087230,0.10996986,5.813767e-01,0.992069372
3,LOC111120925,191.4430,-4.8964652929,1.40924743,8.354541e-06,0.003583072
4,LOC144621283,932.0540,0.0011782772,0.02371156,6.197830e-01,0.992315930
5,LOC144621276,12350.8892,0.0012266111,0.02480930,5.976185e-01,0.992069372
6,LOC111115920,213.3386,0.0676324761,0.25037815,6.160028e-03,0.234887066


# I STOPPED HERE

### DEGs

In [12]:
deg_list <- lapply(file_list, function(df) {
  df %>% filter(abs(log2FoldChange) >= "1" & padj <= 0.05)
})

names(deg_list)
head(deg_list$bb_cc)

[1] "bb_cc" "bc_bb" "bc_cc" "bc_hc" "bc_wc" "bh_ch" "bh_hh" "bw_cw" "bw_ww"
[10] "cb_bb" "cb_bc" "cb_cc" "cb_ch" "cb_cw" "ch_cc" "ch_hc" "cw_cc" "cw_ch"
[19] "cw_wc" "hb_bb" "hb_bh" "hb_cb" "hc_cc" "hc_hh" "hh_cc" "hh_ch" "wb_bb"
[28] "wb_bw" "wb_cb" "wc_cc" "wc_hc" "wc_ww" "wh_hw" "ww_cc" "ww_cw"

,Gene,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,svalue,B1_B1_O01,B1_W5_O50,⋯,W4_W5_G56,W5_B2_G21,W5_C4_G45,W5_H4_G46,W5_W2_G22,W6_B3_G35,W6_B4_G48,W6_H6_G71,W6_W3_G36,W6_W4_G48
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,LOC111102506,948.578114,1.113463,1.3782539,-3.721539,1.980124e-04,4.376733e-02,2.293073e-01,822.0766309,290.844253,⋯,112.13493,230.21305,275.54002,1135.877800,179.4902221,280.52256,796.800632,234.39864,285.80926,204.172325
2,LOC111117765,948.580792,1.019382,0.7028216,-4.035920,5.438871e-05,1.923475e-02,1.031210e-01,1030.2093271,726.100756,⋯,1376.10241,948.80664,354.08763,785.360948,624.4763979,619.93344,1457.901593,607.65903,376.35189,546.966914
3,LOC111104782,61.439122,1.360622,1.1165575,-3.947402,7.900399e-05,2.436630e-02,8.045647e-02,73.1790758,105.027091,⋯,39.19279,32.06539,92.26227,8.127927,183.2296018,26.76742,59.151139,36.65950,33.81713,8.596729
4,LOC144619911,263.266792,-3.777067,3.9685711,3.967465,7.264103e-05,2.349671e-02,1.289212e-01,0.0000000,20.197518,⋯,0.00000,2431.21424,1034.83357,1.015991,0.9348449,0.00000,2.319652,5.55447,4.36350,0.000000
5,LOC111105268,370.537252,-1.988343,2.3808844,3.881096,1.039866e-04,2.795036e-02,1.789722e-01,0.9503776,1852.112359,⋯,17.41902,204.72518,351.59406,42.671617,365.5243586,94.22132,336.349612,1969.61509,644.70715,53.729559
6,LOC111102028,7.951088,21.008371,3.3828306,-6.378646,1.786602e-10,2.961740e-07,6.332811e-07,0.0000000,3.029628,⋯,59.87787,12.33284,0.00000,1.015991,0.0000000,0.00000,0.000000,0.00000,0.00000,4.298365
